# 03 — Prepare Model Data

Joins county-level mean forecast bias with ACS demographic data and
Koppen-Geiger climate classification. Exports a clean, analysis-ready
parquet for the BYM2 model.

**Inputs:**
- Pipeline outputs (via `PipelineDB`): `ifs_bias`, `koppen`
- ACS parquet: `{ACS_DIR}/acs_5yr_{ACS_YEAR}/acs_5yr_{ACS_YEAR}_county.parquet`

**Output:** `analysis/data/model_input.parquet`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..'))
sys.path.insert(0, os.path.join('..', 'scripts'))

import numpy as np
import pandas as pd

from nwp_census_eval.db import PipelineDB
from config import ACS_DIR, ACS_YEAR, ACS_LEVEL

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
# 1. Mean bias per county per lead time
with PipelineDB() as db:
    df_bias = db.query("""
        SELECT
            geo_id,
            lead_time,
            DAYOFYEAR(valid_time) AS day_of_year,
            AVG(bias)              AS mean_bias,
            COUNT(*)               AS n_obs
        FROM ifs_bias
        GROUP BY geo_id, lead_time, DAYOFYEAR(valid_time)
    """)

    # Koppen classification
    if 'koppen' in db.registered_views():
        df_koppen = db.query('SELECT geo_id, category_1 AS koppen_class FROM koppen')
    else:
        df_koppen = pd.DataFrame(columns=['geo_id', 'koppen_class'])

print(f'Bias rows: {len(df_bias):,} | Koppen: {len(df_koppen):,} counties')

In [ ]:
# 2. ACS demographics
acs_path = os.path.join(ACS_DIR, f'acs_5yr_{ACS_YEAR}', f'acs_5yr_{ACS_YEAR}_{ACS_LEVEL}.parquet')
df_acs = pd.read_parquet(acs_path)
print(f'ACS: {df_acs.shape} — columns: {list(df_acs.columns[:10])} ...')

In [ ]:
# 3. Join all together
# Standardise geo_id format (leading zeros for 5-digit FIPS)
df_bias['geo_id'] = df_bias['geo_id'].astype(str).str.zfill(5)
df_acs['geo_id'] = df_acs['geo_id'].astype(str).str.zfill(5) if 'geo_id' in df_acs.columns else df_acs['GEOID'].astype(str).str.zfill(5)

df = df_bias.merge(df_koppen, on='geo_id', how='left')
df = df.merge(df_acs, on='geo_id', how='left')
print(f'Joined: {len(df):,} rows | {df.shape[1]} columns')

In [ ]:
# 4. Save model input
out_path = os.path.join(DATA_DIR, 'model_input.parquet')
df.to_parquet(out_path, index=False)
print(f'Saved {len(df):,} rows → {out_path}')